In [ ]:
# ---- Phase 1: frozen backbone ----
transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

transfer_ckpt_path = str(MODELS_DIR / "transfer_best.keras")
phase1_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-6),
    callbacks.ModelCheckpoint(transfer_ckpt_path, monitor="val_accuracy", mode="max", save_best_only=True),
]

print("=== Phase 1: training classifier head (backbone frozen) ===")
history_phase1 = transfer_model.fit(
    transfer_train_generator,
    validation_data=transfer_val_generator,
    epochs=8,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=phase1_callbacks,
)


In [ ]:
# ---- Phase 2: unfreeze top backbone layers, fine-tune at low LR ----
transfer_base.trainable = True
FINE_TUNE_AT = len(transfer_base.layers) - 30  # unfreeze roughly the last 30 layers only
for layer in transfer_base.layers[:FINE_TUNE_AT]:
    layer.trainable = False

transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),  # 10x lower than Phase 1
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

phase2_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-7),
    callbacks.ModelCheckpoint(transfer_ckpt_path, monitor="val_accuracy", mode="max", save_best_only=True),
]

print(f"=== Phase 2: fine-tuning top {len(transfer_base.layers) - FINE_TUNE_AT} backbone layers ===")
history_phase2 = transfer_model.fit(
    transfer_train_generator,
    validation_data=transfer_val_generator,
    epochs=20,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=phase2_callbacks,
)

# Combine both phases into one continuous history for plotting/reporting
transfer_history = {}
for key in history_phase1.history:
    transfer_history[key] = history_phase1.history[key] + history_phase2.history[key]
pd.DataFrame(transfer_history).to_csv(MODELS_DIR / "transfer_log.csv", index=False)
print(f"\nBest transfer-learning checkpoint saved to: {transfer_ckpt_path}")


In [ ]:
baseline_hist = pd.read_csv(MODELS_DIR / "baseline_log.csv")
transfer_hist = pd.read_csv(MODELS_DIR / "transfer_log.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for hist, label, color in [(baseline_hist, "Baseline CNN", "tab:orange"),
                            (transfer_hist, "Transfer Learning", "tab:blue")]:
    epochs_range = range(1, len(hist) + 1)
    axes[0].plot(epochs_range, hist["loss"], "--", color=color, alpha=0.6, label=f"{label} (train)")
    axes[0].plot(epochs_range, hist["val_loss"], "-", color=color, label=f"{label} (val)")
    axes[1].plot(epochs_range, hist["accuracy"], "--", color=color, alpha=0.6, label=f"{label} (train)")
    axes[1].plot(epochs_range, hist["val_accuracy"], "-", color=color, label=f"{label} (val)")

if len(history_phase1.history["loss"]) > 0:
    axes[0].axvline(len(history_phase1.history["loss"]), color="gray", linestyle=":", label="fine-tune start")
    axes[1].axvline(len(history_phase1.history["loss"]), color="gray", linestyle=":", label="fine-tune start")

axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8)
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.show()


In [ ]:
ABLATION_VARIANTS = {
    "full":            dict(augment=True,  dropout=0.3, class_weighted=True,  lr_schedule=True),
    "no_augmentation": dict(augment=False, dropout=0.3, class_weighted=True,  lr_schedule=True),
    "no_dropout":      dict(augment=True,  dropout=0.0, class_weighted=True,  lr_schedule=True),
    "no_class_weight": dict(augment=True,  dropout=0.3, class_weighted=False, lr_schedule=True),
    "no_lr_schedule":  dict(augment=True,  dropout=0.3, class_weighted=True,  lr_schedule=False),
}
ABLATION_EPOCHS = 8  # short run — controlled comparison, not final training

# FIX: these variants train build_transfer_model() (EfficientNetB0), so — same as Section 8 —
# they must NOT use rescale=1./255. Use non-rescaled generators, mirroring transfer_train_datagen /
# transfer_eval_datagen from Section 5.
no_aug_transfer_datagen = ImageDataGenerator()  # deterministic, no rescale, for the no-augmentation variant

def make_train_generator(augment: bool):
    gen = transfer_train_datagen if augment else no_aug_transfer_datagen
    return gen.flow_from_directory(
        TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=True, seed=SEED,
    )

def run_ablation_variant(name, cfg, epochs=ABLATION_EPOCHS):
    set_seed()
    tl = make_train_generator(cfg["augment"])
    vl = transfer_eval_datagen.flow_from_directory(
        VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False,
    )

    model, base = build_transfer_model(NUM_CLASSES, dropout=cfg["dropout"])
    base.trainable = False
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    cw = CLASS_WEIGHT_DICT if cfg["class_weighted"] else None
    cbs = [callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)] \
          if cfg["lr_schedule"] else []

    hist = model.fit(tl, validation_data=vl, epochs=epochs, class_weight=cw, callbacks=cbs, verbose=1)
    best_acc = max(hist.history["val_accuracy"])

    vl.reset()
    probs = model.predict(vl, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = vl.classes
    best_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return {"variant": name, "best_val_acc": round(float(best_acc), 4),
            "best_val_f1": round(float(best_f1), 4), **cfg}

ablation_results = []
for name, cfg in ABLATION_VARIANTS.items():
    print(f"\n=== Variant: {name} ===")
    ablation_results.append(run_ablation_variant(name, cfg))

ablation_df = pd.DataFrame(ablation_results).sort_values("best_val_f1", ascending=False)
ablation_df.to_csv(RESULTS_DIR / "ablation.csv", index=False)
ablation_df


In [ ]:
# Visualise the marginal contribution of each technique relative to the "full" configuration
full_f1 = ablation_df.loc[ablation_df.variant == "full", "best_val_f1"].values[0]
ablation_df["delta_f1_vs_full"] = ablation_df["best_val_f1"] - full_f1

plt.figure(figsize=(8,5))
order = ablation_df.sort_values("best_val_f1")
plt.barh(order.variant, order.best_val_f1, color="#2b6cb0")
plt.xlabel("Validation macro-F1")
plt.title("Ablation study — effect of removing each technique")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation_chart.png", dpi=150)
plt.show()

print("\nUse delta_f1_vs_full to report each technique's marginal contribution in the report's Results section:")
print(ablation_df[["variant", "best_val_f1", "delta_f1_vs_full"]])
